# Exact TSP Prompt Blocks

This notebook is documentation-only. It reproduces the prompt and interface blocks referenced in the thesis appendix. The runtime code remains in the repository source tree.

Repository: https://github.com/TM-HESSO-202526/llm-tsp-heuristics


## TSP System


In [ ]:
TSP_SYSTEM_PROMPT = '''\
You generate executable Python class-based TSP heuristics. Follow the requested interface exactly. Use only numpy/math in generated code; no external TSP, graph, optimization, or solver libraries.
'''


## Distance Only TSP Initial


In [ ]:
DISTANCE_ONLY_TSP_INITIAL_PROMPT_AND_INTERFACE = '''\
Your task is to design a novel heuristic algorithm for the following TSP optimization problem.

Active objective: Traveling Salesman Problem / permutation tour construction.
Problem:
Given a TSPLIB instance with n cities, return one Hamiltonian tour as a permutation of city indices.
The evaluator closes the tour automatically by adding the edge from the last city back to the first city.
Do not append the start city at the end of the returned tour.

Official evaluation objective:
Minimize total tour length. If an optimum is available, the evaluator reports the percentage gap versus the known optimum.

Problem object seen by your code:
- problem.n: number of cities.
- problem.edge_cost(i, j): true TSPLIB edge-cost query for one edge at a time.
- problem.coords: city coordinates as a numpy array when available.
The full dense distance matrix is not part of the public interface; use bounded edge_cost queries.

Interface:
The generated Python code must define exactly one class named TSPHeuristic:

class TSPHeuristic:
    def __call__(self, problem, rng=None):
        ...

The evaluator will call:

algo = TSPHeuristic()
tour = algo(problem, rng)

Inputs:
- problem is a TSP problem object exposing the interface described above.
- rng is an optional numpy.random.Generator.

Output:
- Return exactly n city indices as an array-like permutation of 0..problem.n-1.
- Each city must appear exactly once.
- Do not append the first city at the end; the evaluator closes the tour itself.
- The algorithm must be self-contained and executable with numpy/math available.

Rules:
- You may use numpy as np, math, lists, dictionaries, sets, and bounded loops.
- Do not import or call sklearn, scipy, pandas, joblib, numba, torch, tensorflow, jax, networkx, OR-Tools, LKH, Concorde, tsplib libraries, multiprocessing, threading, or external optimization libraries.
- Do not read/write files.
- Do not use global hidden state.
- Keep the method scalable for n around 1000 to 1800 cities.
- Avoid cubic or unbounded all-pairs local search.

Objective separation:
The official evaluator computes the tour cost and gap outside your code after your algorithm returns a tour.
No hidden evaluator-side local search, 2-opt, repair, or post-processing is applied beyond validation.
Do not hard-code any instance names, coordinates, optima, reference values, or file paths.

Return format:
# Name:

# Code:
```python
# your code here
```

Generate the first heuristic for this active objective now.
'''


## TSP Historical Family Avoidance


In [ ]:
TSP_HISTORICAL_FAMILY_AVOIDANCE_BLOCK_FOR_DISTANCE_ONLY_RUNS = '''\
Historical family avoidance is ACTIVE.

This run is not only trying to improve the best TSP gap. It is explicitly testing whether the LLM can be pushed away from over-produced heuristic families and generate structurally different constructive mechanisms.

From previous TSP runs, the following families were heavily over-generated and must NOT be used again as the main mechanism:

1. Nearest-neighbor / closest-unvisited constructors
   - Do not build a tour by repeatedly going to the nearest or cheapest next city.
   - Do not disguise this as "adaptive", "hybrid", "enhanced", "priority", or "edge-prior" nearest-neighbor.
2. Cheapest-insertion / regret-insertion constructors
   - Do not construct the tour mainly by inserting one unvisited city into the cheapest position.
   - Do not generate another randomized cheapest insertion, regret insertion, farthest insertion, or sampled insertion variant.
3. Standard 2-opt / relocate / LK-like cleanup
   - Do not produce a base tour and then rely on 2-opt, segment reversal, relocate, swap, or variable-depth exchange.
   - Bounded cleanup is allowed only as a small final repair step, not as the core heuristic.
4. Random restarts / multi-start wrappers
   - Do not create diversity only by trying many random starts of the same known constructor.
   - Randomness is allowed only if the deterministic mechanism is structurally new.

Your next heuristic must choose a genuinely different construction family. Strict novelty requirement:
- The main construction mechanism must be different from nearest-neighbor, cheapest/regret insertion, prior-weighted local edge growth, repeated region/path merging, and 2-opt-centered improvement.
- Do not merely rename an old method.
- Do not just add extra constants, thresholds, restarts, or a final local search to an old family.
- In the generated code comments, briefly indicate the intended mechanism family.

Still obey all TSP interface rules:
- Define exactly one class named TSPHeuristic.
- Return one permutation of 0..problem.n-1.
- Do not append the start city at the end.
- Use only numpy/math/basic Python.
- Do not use external solvers or libraries.
- Keep the method scalable for n around 1000 to 1800.
- Use bounded problem.edge_cost(i, j) queries and problem.coords when helpful; avoid dense all-pairs scans.
'''


## Example TSP Family Focus


In [ ]:
EXAMPLE_TSP_FAMILY_FOCUS_BLOCK_GRID_SECTOR_DECOMPOSITION = '''\
Family-focus mode is ACTIVE.

For the next generated heuristic, you are locked to the following family:

Family id:
grid_sector_decomposition

Family name:
Grid / sector decomposition

Family objective:
Divide the plane into grid cells or angular sectors, order the regions, build local paths, and connect endpoints.

Family description from launcher:
This family is simple and scalable, but previous versions suffered from boundary bugs, weak cell ordering, and bad inter-cell bridges.

Call inside this family block: 1/20

Strict constraints:
- The grid/sector structure must define the macro-order of the tour.
- Use snake-like, radial, sweep, or endpoint-bridging order between cells/sectors.
- Do not just run nearest-neighbor over all cities after computing the cells.
- Fix boundary cases where max-coordinate cities fall outside the last grid cell.
- Local paths and inter-region bridges must both be handled explicitly.
- Your task is to improve this family, not to switch families.
- The declared family must be the main construction mechanism, not a cosmetic wrapper.
- Do not compute the family-specific structure and then ignore it.
- Do not fall back to global nearest-neighbor as the main construction.
- Do not use cheapest/regret insertion as the main construction.
- Do not rely on 2-opt, swaps, relocate moves, or segment reversal as the main source of quality.
- Bounded cleanup is allowed only after the family-specific constructive tour has been built.
- The method must scale to n around 1000 to 1800 cities.
'''


## TSP Family Focus Plan


In [ ]:
FAMILY_FOCUS_PLAN = [
    {
        "id": "mst_skeleton",
        "enabled": False,
        "name": "MST / tree skeleton construction",
        "objective": "Build a sparse tree-like or forest-like skeleton, then traverse, patch, or shortcut it into one Hamiltonian tour.",
        "description": "This family was attempted in historical-family-avoidance runs but often timed out or collapsed into closest-unvisited construction. The goal is to make the tree/skeleton idea real and scalable.",
        "strict_constraints": [
            "The main object must be a spanning-tree-like or forest-like skeleton.",
            "Do not implement this as closest-unvisited nearest-neighbor from the current city.",
            "Avoid dense all-pairs MST construction on 1000+ nodes.",
            "Use bounded local edge queries, coordinate shortlists, or regional tree fragments instead of a full dense graph.",
            "Explicitly explain in code comments how the skeleton is converted into a valid Hamiltonian tour.",
        ],
    },
    {
        "id": "voronoi_regions",
        "enabled": True,
        "name": "Voronoi / region decomposition",
        "objective": "Partition the cities into geometric regions, construct local paths inside each region, and connect region endpoints into one Hamiltonian tour.",
        "description": "Previous runs often computed Voronoi-like regions but then ignored them. Here the region decomposition must actually determine the tour structure.",
        "strict_constraints": [
            "The Voronoi/region partition must actually determine the tour structure.",
            "Do not compute regions and then ignore them.",
            "The algorithm must explicitly solve local ordering inside regions and bridge selection between region endpoints.",
            "Do not fall back to a global closest-unvisited walk after building the regions.",
            "Region-to-region bridges should be selected deliberately, not by simply appending arbitrary region tours.",
        ],
    },
    {
        "id": "convex_hull_outside_in",
        "enabled": True,
        "name": "Convex hull / outside-in insertion",
        "objective": "Start from an outer boundary cycle or approximate hull, then insert interior cities in a structured outside-in way.",
        "description": "Previous hull attempts were either very weak or computed a hull and then switched to nearest-neighbor. This block focuses on making outside-in construction the true mechanism.",
        "strict_constraints": [
            "The main mechanism must be outside-in construction.",
            "Start from an outer boundary cycle, approximate hull, or layered hull structure.",
            "Insert interior cities using insertion delta, edge impact, or geometric distance to tour edges.",
            "Do not switch to nearest-neighbor after computing the hull.",
            "Handle the closing edge of the tour correctly when inserting cities.",
        ],
    },
    {
        "id": "grid_sector_decomposition",
        "enabled": True,
        "name": "Grid / sector decomposition",
        "objective": "Divide the plane into grid cells or angular sectors, order the regions, build local paths, and connect endpoints.",
        "description": "This family is simple and scalable, but previous versions suffered from boundary bugs, weak cell ordering, and bad inter-cell bridges.",
        "strict_constraints": [
            "The grid/sector structure must define the macro-order of the tour.",
            "Use snake-like, radial, sweep, or endpoint-bridging order between cells/sectors.",
            "Do not just run nearest-neighbor over all cities after computing the cells.",
            "Fix boundary cases where max-coordinate cities fall outside the last grid cell.",
            "Local paths and inter-region bridges must both be handled explicitly.",
        ],
    },
    {
        "id": "sparse_geometric_graph",
        "enabled": False,
        "name": "Sparse geometric graph / Gabriel-Delaunay approximation",
        "objective": "Build a bounded-degree geometric graph from coordinates, then construct fragments or paths from that graph and patch them into a tour.",
        "description": "Previous Delaunay/Gabriel attempts were conceptually interesting but often fake, disconnected, or O(n^3). This block tests whether a scalable approximate sparse graph can guide construction.",
        "strict_constraints": [
            "Build a bounded-degree geometric graph using only scalable local approximations.",
            "Do not use scipy, networkx, sklearn, or external triangulation libraries.",
            "Do not scan all triples and do not build an O(n^3) Gabriel graph.",
            "The graph must guide path/fragment construction, not merely be computed and ignored.",
            "Include a fallback for disconnected sparse graph components that preserves the same graph/fragment idea.",
        ],
    },
    {
        "id": "clustering_decomposition",
        "enabled": False,
        "name": "Cluster-based decomposition",
        "objective": "Cluster cities into spatial groups, build local tours or paths in each cluster, then connect clusters through endpoint-aware bridges.",
        "description": "Historical runs often produced cluster names but then used nearest-neighbor inside each cluster and concatenated clusters poorly. This block focuses on real decomposition and bridge quality.",
        "strict_constraints": [
            "The cluster decomposition must be used to define subproblems and bridges.",
            "Do not simply concatenate cluster tours in arbitrary cluster order.",
            "The algorithm must choose cluster order or cluster bridges using endpoint costs or geometric adjacency.",
            "Avoid expensive iterative k-means loops or dense all-pairs assignment when possible.",
            "Do not let the local solver become the whole method; the inter-cluster bridge logic matters.",
        ],
    },
    {
        "id": "spectral_clustering_proxy",
        "enabled": False,
        "name": "Spectral clustering / graph-cut proxy",
        "objective": "Approximate a graph-cut or spectral-style partition without dense eigen-decomposition, then build and bridge region paths.",
        "description": "The LLM attempted spectral clustering before, but dense matrices and eigenvectors were not scalable. This block keeps the partitioning idea while forbidding dense spectral machinery.",
        "strict_constraints": [
            "Do not build a dense n by n distance or affinity matrix.",
            "Do not call numpy eigen-decomposition on an n by n Laplacian.",
            "Use a scalable proxy for spectral/graph-cut behavior, such as coordinate projections, recursive bisection, sparse adjacency, or region cuts.",
            "The partition must feed into local path construction and endpoint bridging.",
            "Do not rename a simple nearest-neighbor tour as spectral clustering.",
        ],
    },
    {
        "id": "polar_angle_sweep",
        "enabled": False,
        "name": "Polar-angle / sweep construction",
        "objective": "Use angular or radial ordering around one or more centers to construct a global sweep tour, with explicit handling of radius jumps and endpoint bridges.",
        "description": "Pure polar sorting was weak in previous runs, but sweep-style construction is scalable and distinct. This block tests whether the LLM can repair the radius-jump problem.",
        "strict_constraints": [
            "The main ordering must be based on angular/radial sweep logic, not closest-unvisited choice.",
            "Handle the common failure where nearby angles but very different radii create long edges.",
            "Consider multi-center sweep, radial bins, alternating sweep direction, or endpoint-aware sector bridges.",
            "Do not sort by angle and then ignore the order by selecting the nearest city anyway.",
            "Bounded cleanup is allowed only after the sweep tour is built.",
        ],
    },
    {
        "id": "region_growth_endpoint_bridging",
        "enabled": False,
        "name": "Region growth / endpoint-bridging construction",
        "objective": "Grow several local path fragments or regions and repeatedly connect endpoints using bridge scarcity, region adjacency, and closure-risk rules.",
        "description": "This is the general region/path-fragment idea that appeared in the avoidance runs. The focus is on endpoint management rather than inserting single cities or walking to nearest neighbors.",
        "strict_constraints": [
            "Maintain multiple open path endpoints or region endpoints during construction.",
            "The main decision must be which endpoints/fragments to connect next, not which single city is closest to the current city.",
            "Avoid subtours and degree violations while building fragments.",
            "Use bridge scarcity, endpoint distance, region adjacency, or future closure risk to choose connections.",
            "Do not reduce this to a single current-city greedy walk.",
        ],
    },
]
